# Modelo Supervisado 2: XGBoost

Este modelo complementa el análisis abordando el objetivo de **clasificar el nivel de riesgo agrícola** (multiclase) de una siembra.
Se utiliza **XGBoost (Extreme Gradient Boosting)** por su robustez ante datos tabulares no lineales y su capacidad para manejar clases desbalanceadas mediante pesos (`sample_weight`).

## Fase 0: Configuración Inicial y Carga de Datos

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    f1_score, accuracy_score, log_loss,
    precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score,
    roc_auc_score, average_precision_score,
    roc_curve, auc, precision_recall_curve
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import label_binarize
from pathlib import Path

from src.data_loader import DataLoader
from src.xgboost_model import XGBoostDataPrep

# Configuraciones globales
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
Path('figuras').mkdir(exist_ok=True)

In [ ]:
# Carga de datos
loader = DataLoader('../data/raw/siap_2010_2024.csv')
try:
    df_raw = loader.cargar()
except FileNotFoundError:
    loader = DataLoader('../siap_procesado.csv')
    df_raw = loader.cargar()

print('Dimensiones del dataset:', df_raw.shape)
df_raw.dropna(subset=['Sembrada', 'Volumenproduccion'], inplace=True)

## Fase 1: Ingeniería de Features y Preparación de Datos

Se aplicará el preprocesamiento definido en `src/xgboost_model.py`:
1. Codificación de variables (LabelEncoding).
2. Creación de variables derivadas (`log_sembrada`, `interaccion_mod_ciclo`).
3. **Split Temporal**: 
   - Entrenamiento: 2010-2020
   - Validación: 2021-2022
   - Test: 2023-2024
4. Cálculo de pesos para el desbalance de clases.

In [ ]:
prep = XGBoostDataPrep(df_raw)
df_processed = prep.preprocess()

(X_train, y_train), (X_val, y_val), (X_test, y_test) = prep.temporal_split()

# Calcular sample weights para entrenamiento debido al desbalance extremo
sample_weights = prep.get_sample_weights(y_train)
target_names = list(prep.riesgo_map.keys())

## Fase 2: Modelo Baseline (XGBoost sin optimizar)

Entrenaremos un clasificador multiclase de XGBoost utilizando sus hiperparámetros por defecto para establecer una métrica de referencia (baseline).

In [ ]:
xgb_baseline = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    random_state=RANDOM_STATE,
    n_estimators=100,
    tree_method='hist'
)

print('Entrenando modelo Baseline...')
xgb_baseline.fit(
    X_train, y_train, 
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)
print('¡Modelo Baseline entrenado!')

In [ ]:
y_val_pred = xgb_baseline.predict(X_val)

print('=== Reporte de Clasificación (Baseline en Validación) ===')
print(classification_report(y_val, y_val_pred, target_names=target_names))

f1_macro_base = f1_score(y_val, y_val_pred, average='macro')
acc_base = accuracy_score(y_val, y_val_pred)
print(f'F1-Score (Macro): {f1_macro_base:.4f}')
print(f'Accuracy: {acc_base:.4f}')

In [ ]:
cm_base = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_title('Matriz de Confusión - Modelo Baseline')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Predicción')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figuras/confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## Fase 3: Regularización con Gamma, Lambda (L2) y Alpha (L1)

XGBoost puede sobreajustarse a los datos de entrenamiento. Para contrarrestar esto, analizaremos el efecto de tres hiperparámetros de regularización:

- **Gamma ($\gamma$, `min_split_loss`)**: Reducción mínima de pérdida requerida para hacer una partición adicional en un nodo hoja. A mayor gamma, el algoritmo se vuelve más conservador, podando ramas que no aportan ganancia suficiente.

$$\text{Ganancia} = \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} - \gamma$$

- **Lambda ($\lambda$, `reg_lambda`, L2)**: Regularización L2 sobre los pesos de las hojas. Suaviza las predicciones reduciendo la magnitud cuadrada de los pesos, lo que disminuye la varianza del modelo.

- **Alpha ($\alpha$, `reg_alpha`, L1)**: Regularización L1 sobre los pesos de las hojas. Puede llevar algunos pesos exactamente a cero, produciendo modelos más simples (*sparsity*).

### Experimento 1: Efecto individual de Gamma

In [ ]:
gamma_values = [0, 0.1, 0.5, 1, 2, 5, 10]

gamma_results = []
for g in gamma_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=g, reg_lambda=1, reg_alpha=0
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    gamma_results.append({
        'gamma': g,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  gamma={g:>5} -> F1-macro={gamma_results[-1]["f1_macro"]:.4f}  logloss={gamma_results[-1]["logloss"]:.4f}')

df_gamma = pd.DataFrame(gamma_results)
print('\nResultados completos:')
display(df_gamma)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_gamma['gamma'], df_gamma['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Gamma en F1-Macro', fontsize=13)
axes[0].set_xlabel('Gamma')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_gamma['gamma'], df_gamma['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Gamma en Log Loss', fontsize=13)
axes[1].set_xlabel('Gamma')
axes[1].set_ylabel('Log Loss')

plt.tight_layout()
plt.savefig('figuras/exp1_gamma.png', dpi=150, bbox_inches='tight')
plt.show()

best_gamma = df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'gamma']
print(f'\nMejor gamma encontrado: {best_gamma}')

### Experimento 2: Efecto individual de Lambda (L2)

In [ ]:
lambda_values = [0, 0.01, 0.1, 1, 5, 10, 50, 100]

lambda_results = []
for l in lambda_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=best_gamma, reg_lambda=l, reg_alpha=0
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    lambda_results.append({
        'reg_lambda': l,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  lambda={l:>6} -> F1-macro={lambda_results[-1]["f1_macro"]:.4f}  logloss={lambda_results[-1]["logloss"]:.4f}')

df_lambda = pd.DataFrame(lambda_results)
print('\nResultados completos:')
display(df_lambda)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_lambda['reg_lambda'], df_lambda['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Lambda (L2) en F1-Macro', fontsize=13)
axes[0].set_xlabel('Lambda')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].set_xscale('symlog', linthresh=0.1)
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_lambda['reg_lambda'], df_lambda['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Lambda (L2) en Log Loss', fontsize=13)
axes[1].set_xlabel('Lambda')
axes[1].set_ylabel('Log Loss')
axes[1].set_xscale('symlog', linthresh=0.1)

plt.tight_layout()
plt.savefig('figuras/exp2_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

best_lambda = df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'reg_lambda']
print(f'\nMejor lambda encontrado: {best_lambda}')

### Experimento 3: Efecto individual de Alpha (L1)

In [ ]:
alpha_values = [0, 0.001, 0.01, 0.1, 1, 5, 10, 50]

alpha_results = []
for a in alpha_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=best_gamma, reg_lambda=best_lambda, reg_alpha=a
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    alpha_results.append({
        'reg_alpha': a,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  alpha={a:>6} -> F1-macro={alpha_results[-1]["f1_macro"]:.4f}  logloss={alpha_results[-1]["logloss"]:.4f}')

df_alpha = pd.DataFrame(alpha_results)
print('\nResultados completos:')
display(df_alpha)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_alpha['reg_alpha'], df_alpha['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Alpha (L1) en F1-Macro', fontsize=13)
axes[0].set_xlabel('Alpha')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].set_xscale('symlog', linthresh=0.001)
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_alpha['reg_alpha'], df_alpha['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Alpha (L1) en Log Loss', fontsize=13)
axes[1].set_xlabel('Alpha')
axes[1].set_ylabel('Log Loss')
axes[1].set_xscale('symlog', linthresh=0.001)

plt.tight_layout()
plt.savefig('figuras/exp3_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

best_alpha = df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'reg_alpha']
print(f'\nMejor alpha encontrado: {best_alpha}')

### Experimento 4: Búsqueda conjunta (RandomizedSearchCV)

Realizamos una búsqueda aleatoria combinando gamma, lambda, alpha junto con otros hiperparámetros clave del modelo.

In [ ]:
param_dist = {
    'gamma': [0, 0.1, 0.5, 1, 2, 5],
    'reg_lambda': [0.01, 0.1, 1, 5, 10, 50],
    'reg_alpha': [0, 0.01, 0.1, 1, 5, 10],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

xgb_search = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    random_state=RANDOM_STATE,
    tree_method='hist'
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=xgb_search,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=True
)

print('Iniciando busqueda de hiperparametros (esto puede tardar varios minutos)...')
random_search.fit(X_train, y_train, sample_weight=sample_weights)

print(f'\nMejor F1-Macro CV: {random_search.best_score_:.4f}')
print(f'Mejores hiperparametros:')
for param, val in random_search.best_params_.items():
    print(f'  {param}: {val}')

In [ ]:
cv_results = pd.DataFrame(random_search.cv_results_)
cols_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
             'param_gamma', 'param_reg_lambda', 'param_reg_alpha',
             'param_max_depth', 'param_learning_rate', 'param_n_estimators']
top10 = cv_results[cols_show].sort_values('rank_test_score').head(10)
print('=== Top 10 configuraciones ===')
display(top10)

### Tabla comparativa de los 4 experimentos

In [ ]:
resumen = pd.DataFrame({
    'Configuracion': ['Baseline (defaults)', 
                      f'Mejor Gamma (g={best_gamma})',
                      f'Mejor Lambda (l={best_lambda})',
                      f'Mejor Alpha (a={best_alpha})',
                      'RandomizedSearchCV (conjunto)'],
    'F1-Macro': [
        f1_macro_base,
        df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'f1_macro'],
        df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'f1_macro'],
        df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'f1_macro'],
        random_search.best_score_
    ],
    'Log Loss': [
        log_loss(y_val, xgb_baseline.predict_proba(X_val)),
        df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'logloss'],
        df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'logloss'],
        df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'logloss'],
        np.nan
    ]
})

print('=== Resumen Comparativo de Regularizacion ===')
display(resumen)

## Fase 4: Modelo Final Optimizado y Evaluación Completa

Entrenamos el modelo con los **mejores hiperparámetros** encontrados en la búsqueda conjunta.
Combinamos los conjuntos de entrenamiento y validación para maximizar la cantidad de datos de entrenamiento,
y evaluamos exclusivamente sobre el conjunto de **test** (2023-2024), datos nunca vistos durante el entrenamiento ni la selección de hiperparámetros.

In [ ]:
# Obtener los mejores hiperparametros del RandomizedSearchCV
best_params = random_search.best_params_.copy()
best_params['objective'] = 'multi:softprob'
best_params['num_class'] = 5
best_params['random_state'] = RANDOM_STATE
best_params['tree_method'] = 'hist'

print('Hiperparametros del modelo final:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

# Combinar Train + Val para entrenamiento final
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
sw_full = prep.get_sample_weights(y_train_full)

# Entrenar modelo final
xgb_final = xgb.XGBClassifier(**best_params)
xgb_final.fit(X_train_full, y_train_full, sample_weight=sw_full, verbose=False)
print('\nModelo final entrenado!')

In [ ]:
# Predicciones en Test
y_test_pred = xgb_final.predict(X_test)
y_test_proba = xgb_final.predict_proba(X_test)

print('=== Reporte de Clasificacion (Test 2023-2024) ===')
print(classification_report(y_test, y_test_pred, target_names=target_names))

# Metricas completas
metrics_final = {
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'F1-Macro': f1_score(y_test, y_test_pred, average='macro'),
    'F1-Weighted': f1_score(y_test, y_test_pred, average='weighted'),
    'Precision-Macro': precision_score(y_test, y_test_pred, average='macro'),
    'Recall-Macro': recall_score(y_test, y_test_pred, average='macro'),
    'Log Loss': log_loss(y_test, y_test_proba),
    'Cohen Kappa': cohen_kappa_score(y_test, y_test_pred),
    'MCC': matthews_corrcoef(y_test, y_test_pred),
    'AUC-ROC (OvR macro)': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
}

print('\n=== Metricas del Modelo Final ===')
for name, val in metrics_final.items():
    print(f'  {name:25s}: {val:.4f}')

In [ ]:
# Matriz de confusion (absoluta y normalizada)
cm_final = confusion_matrix(y_test, y_test_pred)
cm_norm = cm_final.astype('float') / cm_final.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Matriz de Confusion (Absoluta)', fontsize=13)
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Prediccion')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Matriz de Confusion (Normalizada)', fontsize=13)
axes[1].set_ylabel('Valor Real')
axes[1].set_xlabel('Prediccion')

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('figuras/confusion_matrix_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Curvas ROC y Precision-Recall por clase
classes = sorted(y_test.unique())
y_test_bin = label_binarize(y_test, classes=classes)

colores = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=colores[i], linewidth=2,
                 label=f'{target_names[i]} (AUC={roc_auc_val:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('Curvas ROC por Clase (One-vs-Rest)', fontsize=13)
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].legend(loc='lower right', fontsize=8)

for i, cls in enumerate(classes):
    prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_test_proba[:, i])
    pr_auc_val = auc(rec, prec)
    axes[1].plot(rec, prec, color=colores[i], linewidth=2,
                 label=f'{target_names[i]} (AUC-PR={pr_auc_val:.3f})')

axes[1].set_title('Curvas Precision-Recall por Clase', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('figuras/roc_pr_curves_final.png', dpi=150, bbox_inches='tight')
plt.show()

### Comparación Baseline vs. Modelo Optimizado

In [ ]:
# Evaluar baseline tambien en test para comparar
y_test_pred_base = xgb_baseline.predict(X_test)
y_test_proba_base = xgb_baseline.predict_proba(X_test)

comparacion = pd.DataFrame({
    'Metrica': list(metrics_final.keys()),
    'Baseline': [
        accuracy_score(y_test, y_test_pred_base),
        balanced_accuracy_score(y_test, y_test_pred_base),
        f1_score(y_test, y_test_pred_base, average='macro'),
        f1_score(y_test, y_test_pred_base, average='weighted'),
        precision_score(y_test, y_test_pred_base, average='macro'),
        recall_score(y_test, y_test_pred_base, average='macro'),
        log_loss(y_test, y_test_proba_base),
        cohen_kappa_score(y_test, y_test_pred_base),
        matthews_corrcoef(y_test, y_test_pred_base),
        roc_auc_score(y_test, y_test_proba_base, multi_class='ovr', average='macro'),
    ],
    'Optimizado': list(metrics_final.values())
})

comparacion['Delta'] = comparacion['Optimizado'] - comparacion['Baseline']
comparacion['% Mejora'] = ((comparacion['Optimizado'] - comparacion['Baseline']) / comparacion['Baseline'].abs() * 100).round(2)

print('=== Comparacion Baseline vs. Modelo Optimizado (Test) ===')
display(comparacion)

In [ ]:
# Grafico de barras comparativo
metricas_plot = ['Accuracy', 'Balanced Accuracy', 'F1-Macro', 'Precision-Macro', 'Recall-Macro', 'Cohen Kappa']
comp_plot = comparacion[comparacion['Metrica'].isin(metricas_plot)].copy()

x = np.arange(len(metricas_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, comp_plot['Baseline'], width, label='Baseline', color='#c0c0c0', edgecolor='white')
bars2 = ax.bar(x + width/2, comp_plot['Optimizado'], width, label='Optimizado', color='#3b7fc4', edgecolor='white')

ax.set_ylabel('Score')
ax.set_title('Comparacion: Baseline vs. Modelo Optimizado (Test)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metricas_plot, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 1.05)
ax.spines[['top', 'right']].set_visible(False)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('figuras/comparacion_baseline_optimizado.png', dpi=150, bbox_inches='tight')
plt.show()